# Livingston Township — Sales Map

Plots geocoded sales on an interactive Folium map. Default segment: Residential, **2000–present**.

- **Sales (clustered)** — **one pin per location**; its popup lists *every* sale there (date, price, beds/baths). Repeat sales of a parcel and condo units share a coordinate, so grouping per location keeps them all clickable (a one-pin-per-sale map stacks them and only the top is reachable). Pin colored by the latest sale's price quartile.
- **Price heatmap** — density weighted by sale price (per sale, not per location).
- **Median price by parcel** — true parcel polygons (NJOGIS) shaded by median sale price; hover for address / median / # sales.
- **Per-year layers** — one `Sales <year>` checkbox per year, **capped to `CHECKBOX_YEARS` (2020–2026)** so the layer control stays short; the pins/heatmap/choropleth still cover the full 2000+ segment.
- **Search (top-left)** — two boxes: an **address search** over *all* geocoded sales (every year/class, ignores the FILTERS, so any property is findable — it zooms there even if its pin is filtered out), and a **geocoder** that finds *any* typed address/place and drops a marker.

The `FILTERS` cell scopes the visible data (year/class/beds); `CHECKBOX_YEARS` separately caps which per-year checkboxes are generated. Coordinates come from `output/geocode_cache.csv` (run `python3 src/geocode.py` if missing); parcel polygons from `output/parcels_livingston.geojson` (run `python3 src/fetch_parcels.py` if missing). **Re-run all cells** after edits, then open the saved `output/map_*.html` in a browser — the geocoder's custom JS may not run in the inline notebook output sandbox.

In [ ]:
import json
import pandas as pd
import folium
from folium.plugins import MarkerCluster, HeatMap, Search, Geocoder
from folium.features import DivIcon
from pathlib import Path

ROOT = Path('..')
df = pd.read_csv(ROOT / 'data' / 'merged.csv', dtype=str)
cache = pd.read_csv(ROOT / 'output' / 'geocode_cache.csv', dtype={'address': str})

df['sale_date'] = pd.to_datetime(df['Sale Date'], format='%m/%d/%Y', errors='coerce')
df['price'] = pd.to_numeric(df['Sale Price'], errors='coerce')
df['beds']  = pd.to_numeric(df['Beds'], errors='coerce')
df['year']  = df['sale_date'].dt.year
df['addr']  = df['Property Address'].str.strip()

def norm(s):
    # normalize block/lot to match the parcel layer (strip leading zeros)
    s = '' if pd.isna(s) else str(s).strip()
    return s.lstrip('0') if '.' in s else (s.lstrip('0') or '0')

df['block'] = df['Block'].map(norm)
df['lot']   = df['Lot'].map(norm)

# clean (arms-length, dated) + attach coordinates
df = df[df['sale_date'].notna() & (df['price'] >= 1000)]
df = df.merge(cache[cache['matched']][['address', 'lat', 'lon']],
              left_on='addr', right_on='address', how='inner')
print(f'{len(df):,} mappable sales (arms-length, dated, geocoded)')

# parcel polygons for the choropleth layer (run src/fetch_parcels.py if missing)
PARCELS = ROOT / 'output' / 'parcels_livingston.geojson'
parcels = json.loads(PARCELS.read_text()) if PARCELS.exists() else None
print('parcels:', len(parcels['features']) if parcels else 'MISSING - run python3 src/fetch_parcels.py')

## Filters

Same idea as `sales_trends.ipynb`. `None` = no filter. Re-run downward after editing.

In [ ]:
FILTERS = dict(
    property_class="Residential",
    year_min=2000,
    year_max=None,
    beds=None,
)

# Per-year checkbox layers are capped to this range (the FILTERS above still
# control the visible pins / data — e.g. 2000+). Keeps the layer control short.
CHECKBOX_YEARS = range(2020, 2027)

def apply_filters(data, property_class=None, year_min=None, year_max=None, beds=None):
    d = data
    if property_class is not None:
        d = d[d["Property Class"] == property_class]
    if year_min is not None:
        d = d[d["year"] >= year_min]
    if year_max is not None:
        d = d[d["year"] <= year_max]
    if beds is not None:
        d = d[d["beds"] == beds]
    return d

seg = apply_filters(df, **FILTERS)
label = ", ".join(f"{k}={v}" for k, v in FILTERS.items() if v is not None) or "all sales"
print(f"{len(seg):,} sales in segment: {label}")

## Build the map

Marker color buckets the sale price (green = lower, red = higher) relative to the segment's quartiles.

In [ ]:
q1, q2, q3 = seg['price'].quantile([0.25, 0.5, 0.75])
def color(p):
    return ('green' if p < q1 else 'lightgreen' if p < q2
            else 'orange' if p < q3 else 'red')

def dot(c):
    # Colored dot as a DivIcon Marker. We use folium.Marker (an L.marker) rather
    # than CircleMarker because MarkerCluster's spiderfy only repositions
    # L.marker on overlap; CircleMarkers stay stacked and only the top is
    # clickable (the bug where two close pins couldn't both be opened).
    return DivIcon(icon_size=(14, 14), icon_anchor=(7, 7),
                   html=f'<div style="width:12px;height:12px;border-radius:50%;'
                        f'background:{c};border:1px solid #333;opacity:0.85"></div>')

def location_popup(g):
    # g = all sales at one coordinate, ascending by date. Many coordinates hold
    # several sales (repeat sales of a parcel, or condo units that geocode to the
    # same building point) -> list every sale so none is hidden under another pin.
    multi_addr = g['addr'].nunique() > 1
    lines = [(f"{r['addr']} &middot; " if multi_addr else "") +
             f"{r['sale_date']:%Y-%m-%d}: ${r['price']:,.0f} "
             f"({r['Beds']}bd/{r['Baths']}ba)" for _, r in g.iterrows()]
    title = (g['addr'].iloc[0] if not multi_addr
             else f"{g['addr'].nunique()} addresses / {len(g)} sales")
    sub = (f"Zoning: {g['Zoning'].iloc[-1]} &middot; Acreage: {g['Acreage'].iloc[-1]}"
           if not multi_addr else '')
    return f"<b>{title}</b><br>{sub}{'<br>' if sub else ''}" + "<br>".join(lines)

center = [seg['lat'].median(), seg['lon'].median()]
m = folium.Map(location=center, zoom_start=14, tiles='cartodbpositron')

# --- layer 1: one pin per LOCATION (clustered); popup lists all sales there ---
# Grouping by coordinate avoids stacked markers where only the top one is
# clickable (most sale points share a coordinate with at least one other sale).
cluster = MarkerCluster(name='Sales (clustered)').add_to(m)
for (lat, lon), g in seg.groupby(['lat', 'lon']):
    g = g.sort_values('sale_date')
    folium.Marker(
        [lat, lon], icon=dot(color(g['price'].iloc[-1])),  # color by latest sale
        popup=folium.Popup(location_popup(g), max_width=300),
    ).add_to(cluster)

# --- layer 2: price heatmap ---
heat = [[r['lat'], r['lon'], r['price']] for _, r in seg.iterrows()]
HeatMap(heat, name='Price heatmap', radius=15, blur=20, show=False).add_to(m)

# --- layer 3: parcel choropleth (median price per lot, segment only) ---
if parcels:
    agg = (seg.groupby(['block', 'lot'])
              .agg(median_price=('price', 'median'), sales=('price', 'size'))
              .reset_index())
    agg['key'] = agg['block'] + '_' + agg['lot']
    stats = agg.set_index('key')[['median_price', 'sales']].to_dict('index')

    feats = []
    for f in parcels['features']:
        k = f"{f['properties']['block']}_{f['properties']['lot']}"
        if k in stats:
            f = {**f, 'properties': {**f['properties'], **stats[k]}}
            feats.append(f)
    print(f'{len(feats)} parcels with sales in segment shaded')

    lo, hi = agg['median_price'].quantile([0.05, 0.95])
    import branca.colormap as cm
    cmap = cm.linear.YlOrRd_09.scale(lo, hi)
    cmap.caption = 'Median sale price ($)'

    def style(feature):
        return {'fillColor': cmap(feature['properties']['median_price']),
                'color': '#555', 'weight': 0.5, 'fillOpacity': 0.7}

    folium.GeoJson(
        {'type': 'FeatureCollection', 'features': feats},
        name='Median price by parcel', style_function=style, show=False,
        tooltip=folium.GeoJsonTooltip(
            fields=['PROP_LOC', 'median_price', 'sales'],
            aliases=['Address', 'Median price', '# sales'], localize=True),
    ).add_to(m)
    cmap.add_to(m)

# --- per-year layers: one checkbox per year (capped to CHECKBOX_YEARS) ---
# Tick a year to show its sales. All off by default. The pins/heatmap/choropleth
# above still cover the whole segment (2000+); these checkboxes just isolate
# recent years without flooding the layer control with 20+ entries. Grouped by
# coordinate too, so within a year stacked sales stay clickable.
for yr in CHECKBOX_YEARS:
    yr_sales = seg[seg['year'] == yr]
    if yr_sales.empty:
        continue
    fg = folium.FeatureGroup(name=f'Sales {yr}', show=False)
    for (lat, lon), g in yr_sales.groupby(['lat', 'lon']):
        g = g.sort_values('sale_date')
        folium.Marker(
            [lat, lon], icon=dot(color(g['price'].iloc[-1])),
            popup=folium.Popup(location_popup(g), max_width=300),
        ).add_to(fg)
    fg.add_to(m)

# --- search box 1: find ANY geocoded sale, regardless of the FILTERS ---
# Built from the full geocoded dataset (df), not the filtered segment, so a
# property hidden by the current filters (e.g. 77 Sycamore, a 2013 sale, when
# year_min=2020) is still findable. Selecting it zooms to the property and shows
# its latest sale; the matching pin may not be visible unless filters include it.
latest = df.sort_values('sale_date').groupby('addr', as_index=False).last()
search_gj = folium.GeoJson(
    {'type': 'FeatureCollection', 'features': [
        {'type': 'Feature',
         'geometry': {'type': 'Point', 'coordinates': [r['lon'], r['lat']]},
         'properties': {'addr': r['addr'],
                        'latest': f"${r['price']:,.0f} ({r['sale_date']:%Y})"}}
        for _, r in latest.iterrows()]},
    name='Searchable sales (all years)', show=False,
    marker=folium.CircleMarker(radius=6, color='blue', fill=True, fill_opacity=0.9),
    popup=folium.GeoJsonPopup(fields=['addr', 'latest'],
                              aliases=['', 'latest sale:']),
).add_to(m)
Search(layer=search_gj, search_label='addr', position='topleft', collapsed=False,
       placeholder='Search any sale address...').add_to(m)

# --- search box 2: geocode ANY address and drop a marker (in the data or not) ---
Geocoder(position='topleft', collapsed=False, add_marker=True,
         provider='nominatim').add_to(m)

folium.LayerControl(collapsed=False).add_to(m)
m

## Save standalone HTML

In [ ]:
out = ROOT / "output"
out.mkdir(exist_ok=True)
slug = label.replace(", ", "_").replace("=", "-").replace(" ", "")
path = out / f"map_{slug}.html"
m.save(str(path))
print("wrote", path)